In [1]:
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset, get_dataset_config_names
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from transformers import Trainer, DataCollatorWithPadding, TrainingArguments, pipeline
from collections import namedtuple
import torch.nn as nn
import torch.nn.functional as F
import urllib.request
import torch
import torchmetrics
import huggingface_hub
import tokenizers
import transformers

device = 'cuda'

/home/damian/New Folder/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:
def download_shakespaere():
    path = Path("datasets2/shakespeare/shakespeare.txt")
    if not path.is_file():
        path.parent.mkdir(parents=True, exist_ok=True)
        url = "https://homl.info/shakespeare"
        urllib.request.urlretrieve(url, path)
    return path.read_text()

shakespeare_text = download_shakespaere()

In [ ]:
print(shakespeare_text[:80])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


In [ ]:
vocab = sorted(set(shakespeare_text.lower()))
"".join(vocab)

"\n !$&',-.3:;?abcdefghijklmnopqrstuvwxyz"

In [ ]:
char_to_idx = {char : index for index, char in enumerate(vocab)}
idx_to_char = {index : char for index, char in enumerate(vocab)}

print(char_to_idx['a'])
print(idx_to_char[13])

13
a


In [ ]:
def encode(text):
    return torch.tensor([char_to_idx[char] for char in text.lower()])

def decode(idx):
    return "".join([idx_to_char[index.item()] for index in idx])

In [ ]:
val_text = "Hello, world!"
encoded_text = encode(val_text)
print(encoded_text)
decoded_text = decode(encoded_text)
print(decoded_text)

tensor([20, 17, 24, 24, 27,  6,  1, 35, 27, 30, 24, 16,  2])
hello, world!


In [ ]:
class CharDataset(Dataset):
    def __init__(self, text, window_length):
        super().__init__()
        self.encoded_text = encode(text)
        self.window_length = window_length
        
    def __len__(self):
        return len(self.encoded_text) - self.window_length
    
    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError('dataset index is out of range')
        end = self.window_length + idx
        window = self.encoded_text[idx:end]
        target = self.encoded_text[idx+1:end+1]
        return window, target

In [ ]:
window_length = 32
batch_size = 256

train_set = CharDataset(shakespeare_text[:1000000], window_length)
valid_set = CharDataset(shakespeare_text[1000000:1060000], window_length)
test_set  = CharDataset(shakespeare_text[1060000:], window_length)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size)
test_loader  = DataLoader(test_set, batch_size=batch_size)

In [ ]:
torch.manual_seed(52)
embed = nn.Embedding(5, 3)
embed(torch.tensor([[3, 2], [0, 2]]))

tensor([[[-0.3845, -0.3752, -0.6016],
         [-0.7003,  0.7729,  0.6928]],

        [[-0.1455, -1.1507, -0.6676],
         [-0.7003,  0.7729,  0.6928]]], grad_fn=<EmbeddingBackward0>)

In [ ]:
class ShakespeareModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=10, hidden_dim=128, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, n_layers, batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, X):
        embeddings = self.embed(X)
        outputs, _states = self.gru(embeddings)
        return self.output(outputs).permute(0, 2, 1)

In [ ]:
def eval_model(model, metric, data_loader):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
        return metric.compute()
    
def train_model(model, optimizer, criterion, train_loader, valid_loader, metric, n_epochs=50, patience=10, factor=0.1):
    history = {'train_losses':[], 'train_metrics':[], 'valid_metrics':[]}
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, mode='min', factor=factor, patience=patience)
    for epoch in range(n_epochs):
        model.train()
        metric.reset()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        history['train_losses'].append(total_loss/len(train_loader))
        history['train_metrics'].append(metric.compute().item())
        val_score = eval_model(model, metric, valid_loader).item()
        history['valid_metrics'].append(val_score)
        scheduler.step(val_score)
        
        print(f'Epoch: {epoch+1}\tTrain loss: {history["train_losses"][-1]:.3f}\tTrain metrics: {history["train_metrics"][-1]:.3f}\tValid metrics: {history["valid_metrics"][-1]:.3f}')
        
    return history

In [ ]:
model_1 = ShakespeareModel(len(vocab)).to('cuda')

optimizer = torch.optim.NAdam(params=model_1.parameters())
xentropy = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=len(vocab)).to('cuda')

train_model(model_1, optimizer, xentropy, train_loader, valid_loader, accuracy, 10)

Epoch: 1	Train loss: 1.572	Train metrics: 0.520	Valid metrics: 0.523
Epoch: 2	Train loss: 1.426	Train metrics: 0.555	Valid metrics: 0.531
Epoch: 3	Train loss: 1.406	Train metrics: 0.560	Valid metrics: 0.531
Epoch: 4	Train loss: 1.396	Train metrics: 0.562	Valid metrics: 0.533
Epoch: 5	Train loss: 1.390	Train metrics: 0.563	Valid metrics: 0.534
Epoch: 6	Train loss: 1.386	Train metrics: 0.564	Valid metrics: 0.536
Epoch: 7	Train loss: 1.383	Train metrics: 0.565	Valid metrics: 0.534
Epoch: 8	Train loss: 1.381	Train metrics: 0.565	Valid metrics: 0.535
Epoch: 9	Train loss: 1.379	Train metrics: 0.566	Valid metrics: 0.535
Epoch: 10	Train loss: 1.377	Train metrics: 0.566	Valid metrics: 0.536


{'train_losses': [1.5717941543651353,
  1.4261526298974523,
  1.405701230485817,
  1.3960244198543568,
  1.390194006670397,
  1.386164004734968,
  1.3830708133799052,
  1.3807898189272758,
  1.3788103970963845,
  1.3770832329233313],
 'train_metrics': [0.5196436643600464,
  0.5548205375671387,
  0.5596721172332764,
  0.5619174838066101,
  0.5633241534233093,
  0.5643031597137451,
  0.5649628043174744,
  0.5654542446136475,
  0.565984845161438,
  0.5663533806800842],
 'valid_metrics': [0.5226203799247742,
  0.5308961868286133,
  0.5309737920761108,
  0.5331098437309265,
  0.5343177914619446,
  0.5359967947006226,
  0.5344569087028503,
  0.53451007604599,
  0.5352240204811096,
  0.5363589525222778]}

In [ ]:
model_1.eval()
text='To be or not to b'
encoded_text_2 = encode(text).unsqueeze(dim=0).to('cuda')
with torch.no_grad():
    Y_logits = model_1(encoded_text_2)
    print(Y_logits.shape)
    predicted_char_id = Y_logits[0, :, -1].argmax().item()
    predicted_char = idx_to_char[predicted_char_id]
    
print(predicted_char)

torch.Size([1, 39, 17])
e


In [ ]:
probs = torch.tensor([0.5, 0.4, 0.1])
torch.multinomial(input=probs, num_samples=10, replacement=True)

tensor([1, 1, 0, 0, 2, 0, 0, 0, 0, 1])

In [ ]:
def next_char(model, text, temperature=1):
    encoded_text = encode(text).unsqueeze(dim=0).to(device)
    with torch.no_grad():
        Y_logits = model(encoded_text)
        Y_probas = F.softmax(Y_logits[0, :, -1]/temperature, dim=-1)
        predicted_char_id = torch.multinomial(input=Y_probas, num_samples=1).item()
    return idx_to_char[predicted_char_id]

def extend_text(model, text, n_chars=80, temperature=1):
    for _ in range(n_chars):
        text += next_char(model, text, temperature)
    return text

In [ ]:
print(extend_text(model_1, 'To be or not to be', n_chars=200, temperature=1))
print(extend_text(model_1, 'To be or not to be', n_chars=200, temperature=0.01))
print(extend_text(model_1, 'To be or not to be', n_chars=200, temperature=0.4))
print(extend_text(model_1, 'To be or not to be', n_chars=200, temperature=1.4))

To be or not to be the best.

hastings:
i shall dreams of
his perforce,
even has
my feather
than you go withdraw you so, 'all you to command a trencted
would live of the while he and their hands like sweet which all; n
To be or not to be so shall be so shall be so shall be so shall be so shall be so shall be so shall be so shall be so shall be so shall be so shall be so shall be so shall be so shall be so shall be so shall be so shal
To be or not to be the county hands and the world to the wind so fair cousin, and thou shalt have the people, and therefore make him since thou hast so much more than you have been so dear lordship on the world be brin
To be or not to be part coloos.
for us right me whaps, comest so ve of quameous sixter of griefs:
gently reverend'st def!
they berwheves, seemerge evil,
i, my nothing shrow reguert to smilk hath nice
truly ho!
rejerior


### Tokenization

In [ ]:
imdb_dataset = load_dataset('imdb')
split = imdb_dataset['train'].train_test_split(train_size=0.8, shuffle=True)
imdb_train_set, imdb_valid_set = split['train'], split['test']
imdb_test_set = imdb_dataset['test']

Using the latest cached version of the dataset since imdb couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'plain_text' at /home/damian/.cache/huggingface/datasets/imdb/plain_text/0.0.0/e6281661ce1c48d982bc483cf8a173c1bbeb5d31 (last modified on Thu Jul 23 21:42:43 2026).


In [ ]:
print(imdb_train_set[0]['label'])
print(imdb_train_set[0]['text'])
print(imdb_train_set[1]['label'])
print(imdb_train_set[1]['text'])

1
I saw Soylent Green back in 1973 when it was first released and maybe another eight times over the years on T.V. or video. It was always one of my favorite sci-fi and/or Charlton Heston films.<br /><br />Recently, the Egyptian theater in L.A. had a twelve film Charlton Heston retrospective. I flew in from out of state to see six of the films over a two day period. Soylent Green looked great on the large Egyptian screen with a perfect new print. From its opening montage to the going home scene to the great ending the film was fantastic.<br /><br />Charlton Heston as a cop who lives in a dog eat dog world with few natural resources left and no understanding as to how the world used to be and Eddie Robinson as a man who remembers the past are both great.<br /><br />Their chemistry together is wonderful. The film also looks so much better in a great 35mm print. Fleisher really knows how to fill the screen,and the cinematoraphy, writing, music used, and everything about it works. The film

In [ ]:
bpe_model = tokenizers.models.BPE(unk_token="<unk>")
bpe_tokenizer = tokenizers.Tokenizer(bpe_model)
bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
special_tokens = ["<pad>", "<unk>"]
bpe_trainer = tokenizers.trainers.BpeTrainer(vocab_size=1000, special_tokens=special_tokens)
train_reviews = [review['text'].lower() for review in imdb_train_set]
bpe_tokenizer.train_from_iterator(train_reviews, bpe_trainer)

In [ ]:
some_text = "what an awesome movie! 😀, my name is damian"

bpe_encoded = bpe_tokenizer.encode(some_text)
print(bpe_encoded)
print(bpe_encoded.tokens)
print(bpe_encoded.ids)

Encoding(num_tokens=16, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])
['what', 'an', 'aw', 'es', 'ome', 'movie', '!', '<unk>', ',', 'my', 'n', 'ame', 'is', 'd', 'am', 'ian']
[301, 137, 372, 147, 222, 209, 3, 1, 14, 304, 54, 356, 139, 44, 187, 596]


In [ ]:
print(bpe_tokenizer.decode(bpe_encoded.ids))
bpe_encoded.offsets

what an aw es ome movie ! , my n ame is d am ian


[(0, 4),
 (5, 7),
 (8, 10),
 (10, 12),
 (12, 15),
 (16, 21),
 (21, 22),
 (23, 24),
 (24, 25),
 (26, 28),
 (29, 30),
 (30, 33),
 (34, 36),
 (37, 38),
 (38, 40),
 (40, 43)]

In [ ]:
bpe_tokenizer.encode_batch(train_reviews[:3])

[Encoding(num_tokens=538, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=552, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=977, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])]

In [ ]:
bpe_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
bpe_tokenizer.enable_truncation(max_length=500)

In [ ]:
bpe_encodings = bpe_tokenizer.encode_batch(train_reviews[:3])
bpe_ids = torch.tensor([encoding.ids for encoding in bpe_encodings])
print(bpe_ids.shape)
bpe_ids

torch.Size([3, 500])


tensor([[ 49, 758, 215,  ...,  49, 187, 263],
        [136, 642, 138,  ...,   3,  30, 160],
        [432,  14,  49,  ..., 603, 154,  19]])

In [ ]:
attention_mask = torch.tensor([encoding.attention_mask for encoding in bpe_encodings])
print(attention_mask)
print(attention_mask.sum(dim=-1))

tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]])
tensor([500, 500, 500])


#### Pretrained tokenizers

In [ ]:
#gpt2 BBpe
gpt2_tokenizer = transformers.AutoTokenizer.from_pretrained('gpt2')
gpt2_encode = gpt2_tokenizer(train_reviews[:3], truncation=True, max_length=500)

In [ ]:
print(gpt2_encode['input_ids'][0][:10])
print(gpt2_tokenizer.decode(gpt2_encode['input_ids'][0][:10]))

[72, 2497, 523, 2645, 298, 4077, 736, 287, 15674, 618]
i saw soylent green back in 1973 when


In [ ]:
#bert WordPiece
bert_tokenizer = transformers.AutoTokenizer.from_pretrained('bert-base-uncased')
bert_encodings = bert_tokenizer(train_reviews[:3], truncation=True, padding=True, max_length=500, return_tensors='pt')

In [ ]:
print(bert_encodings['input_ids'])
print(bert_encodings['attention_mask'])

tensor([[ 101, 2023, 3185,  ...,    0,    0,    0],
        [ 101, 2417, 3239,  ..., 1005, 2828,  102],
        [ 101, 1045, 2245,  ...,    0,    0,    0]])
tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]])


In [ ]:
#albert Unigram
albert_tokenizer = transformers.AutoTokenizer.from_pretrained('albert-base-v2')
albert_encodings = albert_tokenizer(train_reviews[:3], padding=True, truncation=True, max_length=500, return_tensors='pt')

In [ ]:
#a wrap
hf_tokenizer = transformers.PreTrainedTokenizerFast(tokenizer_object=bpe_tokenizer)
#hf_encodings = hf_tokenizer(train_reviews[:3], truncation=True, padding=True, return_tensors='pt', max_length=500)

In [ ]:
def collate_fn(batch, tokenizer=bert_tokenizer):
    reviews = [review['text'] for review in batch]
    labels  = [[review['label']] for review in batch]
    encodings = bert_tokenizer(reviews, padding=True, truncation=True, max_length=200, return_tensors='pt')
    labels = torch.tensor(labels, dtype=torch.float32)
    return encodings, labels

batch_size = 128
imdb_train_loader = DataLoader(imdb_train_set, batch_size=batch_size, collate_fn=collate_fn, shuffle=True)
imdb_valid_loader = DataLoader(imdb_valid_set, batch_size=batch_size, collate_fn=collate_fn)
imdb_test_loader  = DataLoader(imdb_test_set,  batch_size=batch_size, collate_fn=collate_fn)

In [ ]:
bert_tokenizer.vocab_size

30522

In [ ]:
class SentimentAnalysisModel(nn.Module):
    def __init__(self, vocab_size, layers=2, embed_dim=128, hidden_dim=64, pad_id=0, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=layers, batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, 1)
        
    def forward(self, encodings):
        embeds = self.embed(encodings['input_ids'])
        length = encodings['attention_mask'].sum(dim=1)
        packed = pack_padded_sequence(embeds, lengths=length.cpu(), batch_first=True, enforce_sorted=False)
        _outputs, hidden_states = self.gru(packed)
        return self.output(hidden_states[-1])

In [ ]:
model_2 = SentimentAnalysisModel(vocab_size=bert_tokenizer.vocab_size).to(device)

optimizer = torch.optim.NAdam(params=model_2.parameters())# does not work #upd: It started working, spettacolo!
#optimizer = torch.optim.SGD(params=model_2.parameters(), lr=0.002, momentum=0.9, nesterov=True) # 54%
#optimizer = torch.optim.RMSprop(params=model_2.parameters()) does not work
binary_loss = nn.BCEWithLogitsLoss()
metric = torchmetrics.Accuracy(task='binary').to(device)

history = train_model(model_2, optimizer, binary_loss, imdb_train_loader, imdb_valid_loader, metric, 10)

Epoch: 1	Train loss: 0.652	Train metrics: 0.609	Valid metrics: 0.736
Epoch: 2	Train loss: 0.447	Train metrics: 0.793	Valid metrics: 0.835
Epoch: 3	Train loss: 0.279	Train metrics: 0.886	Valid metrics: 0.845
Epoch: 4	Train loss: 0.188	Train metrics: 0.931	Valid metrics: 0.843
Epoch: 5	Train loss: 0.118	Train metrics: 0.960	Valid metrics: 0.816
Epoch: 6	Train loss: 0.068	Train metrics: 0.980	Valid metrics: 0.851
Epoch: 7	Train loss: 0.051	Train metrics: 0.984	Valid metrics: 0.853
Epoch: 8	Train loss: 0.036	Train metrics: 0.991	Valid metrics: 0.849
Epoch: 9	Train loss: 0.019	Train metrics: 0.995	Valid metrics: 0.848
Epoch: 10	Train loss: 0.014	Train metrics: 0.997	Valid metrics: 0.847


In [ ]:
sequences = torch.tensor([[1,2,0,0], [5,6,7,8]])
packed = pack_padded_sequence(sequences, lengths=[2, 4], enforce_sorted=False, batch_first=True)
print(packed)
padded, length = pad_packed_sequence(packed, batch_first=True)
print(padded, length)

PackedSequence(data=tensor([5, 1, 6, 2, 7, 8]), batch_sizes=tensor([2, 2, 1, 1]), sorted_indices=tensor([1, 0]), unsorted_indices=tensor([1, 0]))
tensor([[1, 2, 0, 0],
        [5, 6, 7, 8]]) tensor([2, 4])


In [ ]:
#bidim
class SentimentAnalysisModelBi(nn.Module):
    def __init__(self, vocab_size, layers=2, embed_dim=128, hidden_dim=64, pad_id=0, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=layers, batch_first=True, dropout=dropout, bidirectional=True)
        self.output = nn.Linear(hidden_dim * 2, 1)
        
    def forward(self, encodings):
        embeds = self.embed(encodings['input_ids'])
        length = encodings['attention_mask'].sum(dim=1)
        packed = pack_padded_sequence(embeds, lengths=length.cpu(), batch_first=True, enforce_sorted=False)
        _outputs, hidden_states = self.gru(packed)
        n_dims = self.output.in_features
        top_states = hidden_states[-2:].remute(1,0,2).reshape(-1, n_dims)
        return self.output(top_states)

##### Pretrained embedding models

In [ ]:
bert_model = transformers.AutoModel.from_pretrained('bert-base-uncased')
bert_model.embeddings.word_embeddings

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3518.55it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding(30522, 768, padding_idx=0)

In [ ]:
class SentimentAnalysisModelPretrained(nn.Module):
    def __init__(self, pretrained_embeddings, layers=2, embed_dim=128, hidden_dim=64, dropout=0.2):
        super().__init__()
        weights = pretrained_embeddings.weight.data
        self.embed = nn.Embedding.from_pretrained(weights, freeze=True)
        embed_dim = weights.shape[-1]
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=layers, batch_first=True, dropout=dropout, bidirectional=True)
        self.output = nn.Linear(hidden_dim * 2, 1)
        
    def forward(self, encodings):
        embeds = self.embed(encodings['input_ids'])
        length = encodings['attention_mask'].sum(dim=1)
        packed = pack_padded_sequence(embeds, lengths=length.cpu(), batch_first=True, enforce_sorted=False)
        _outputs, hidden_states = self.gru(packed)
        n_dims = self.output.in_features
        top_states = hidden_states[-2:].remute(1,0,2).reshape(-1, n_dims)
        return self.output(top_states)

In [ ]:
bert_encodings = bert_tokenizer(train_reviews[:3], max_length=200, padding=True, truncation=True, return_tensors='pt')
bert_output = bert_model(**bert_encodings)
bert_output.last_hidden_state.shape

torch.Size([3, 200, 768])

In [ ]:
class SentimentAnalysisBert(nn.Module):
    def __init__(self, n_layers=2, hidden_dim=64, dropout=0.2):
        super().__init__()
        self.bert = transformers.AutoModel.from_pretrained('bert-base-uncased')
        embed_dim = self.bert.config.hidden_size
        self.gru = nn.GRU(embed_dim, hidden_dim, n_layers, batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, 1)
        
    def forward(self, X):
        contextualized_embeddings = self.bert(**X).last_hidden_state
        lengths = X['attention_mask'].sum(dim=1)
        packed = pack_padded_sequence(contextualized_embeddings, lengths=lengths.cpu(), enforce_sorted=False, batch_first=True)
        _outputs, hidden_states = self.gru(packed)
        return self.output(hidden_states[-1])

In [ ]:
model_3 = SentimentAnalysisBert().to(device)
model_3.bert.requires_grad_(False)

optimizer = torch.optim.NAdam(params=model_3.parameters())
binary_loss = nn.BCEWithLogitsLoss()
metric = torchmetrics.Accuracy(task='binary').to(device)

history = train_model(model_3, optimizer, binary_loss, imdb_train_loader, imdb_valid_loader, metric, 5)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9905.73it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch: 1	Train loss: 0.429	Train metrics: 0.795	Valid metrics: 0.827
Epoch: 2	Train loss: 0.284	Train metrics: 0.882	Valid metrics: 0.885
Epoch: 3	Train loss: 0.258	Train metrics: 0.892	Valid metrics: 0.885
Epoch: 4	Train loss: 0.230	Train metrics: 0.905	Valid metrics: 0.889
Epoch: 5	Train loss: 0.211	Train metrics: 0.915	Valid metrics: 0.890


In [ ]:
class SentimentAnalysisBert2(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = transformers.AutoModel.from_pretrained('bert-base-uncased')
        embed_dim = self.bert.config.hidden_size
        self.output = nn.Linear(embed_dim, 1)
        
    def forward(self, X):
        bert_output = self.bert(**X)
        return self.output(bert_output.last_hidden_state[:, 0])

In [ ]:
model_4 = SentimentAnalysisBert2().to(device)
model_4.bert.requires_grad_(False)

optimizer = torch.optim.NAdam(params=model_4.parameters())
binary_loss = nn.BCEWithLogitsLoss()
metric = torchmetrics.Accuracy(task='binary').to(device)

history = train_model(model_4, optimizer, binary_loss, imdb_train_loader, imdb_valid_loader, metric, 5)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11266.94it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch: 1	Train loss: 0.517	Train metrics: 0.766	Valid metrics: 0.811
Epoch: 2	Train loss: 0.439	Train metrics: 0.802	Valid metrics: 0.819
Epoch: 3	Train loss: 0.421	Train metrics: 0.812	Valid metrics: 0.812
Epoch: 4	Train loss: 0.415	Train metrics: 0.814	Valid metrics: 0.819
Epoch: 5	Train loss: 0.407	Train metrics: 0.818	Valid metrics: 0.829


In [ ]:
best_for_binary_csf = transformers.BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2, dtype=torch.float16).to(device)
encoding = bert_tokenizer(["This was a great movie!"])
with torch.no_grad():
    output = best_for_binary_csf(input_ids=torch.tensor(encoding['input_ids'], device=device), 
                                 attention_mask=torch.tensor(encoding['attention_mask'], device=device),
                                 labels=torch.tensor([1], device=device))
    
print(output.logits)
print(torch.softmax(output.logits, dim=-1))
print(output.loss)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2662.76it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

tensor([[ 0.3054, -0.0172]], device='cuda:0', dtype=torch.float16)
tensor([[0.5801, 0.4202]], device='cuda:0', dtype=torch.float16)
tensor(0.8672, device='cuda:0', dtype=torch.float16)


##### Trainer API

In [ ]:
def tokenize(batch):
    return bert_tokenizer(batch['text'], truncation=True, max_length=200)

tok_imdb_train_set = imdb_train_set.map(tokenize, batched=True)
tok_imdb_valid_set = imdb_valid_set.map(tokenize, batched=True)
tok_imdb_test_set  = imdb_test_set.map(tokenize, batched=True)

Map: 100%|██████████| 25000/25000 [00:03<00:00, 8154.89 examples/s]


In [ ]:
def compute_accuracy(pred):
    return{'accuracy' : (pred.label_ids == pred.predictions.argmax(-1)).mean()}

In [ ]:
train_args = transformers.TrainingArguments(output_dir='my_imdb_model', num_train_epochs=2, per_device_train_batch_size=64,
                                            per_device_eval_batch_size=64, eval_strategy='epoch', logging_strategy='epoch',
                                            save_strategy='epoch', load_best_model_at_end=True, metric_for_best_model='accuracy',
                                            report_to='none')


In [ ]:
trainer = Trainer(best_for_binary_csf, train_args, train_dataset=tok_imdb_train_set, eval_dataset=tok_imdb_valid_set,
                  compute_metrics=compute_accuracy, data_collator=DataCollatorWithPadding(bert_tokenizer))
train_output = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.775040,nan,0.505200
2,0.000000,nan,0.505200


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.57it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

In [ ]:
best_for_binary_csf.config

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float16",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "problem_type": "single_label_classification",
  "tie_word_embeddings": true,
  "transformers_version": "5.14.1",
  "type_vocab_size": 2,
  "use_cache": false,
  "vocab_size": 30522
}

##### Pipeline

In [ ]:
classifier_imdb = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english',
                           truncation=True, max_length=512)
classifier_imdb(train_reviews[:10])

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 11415.76it/s]


[{'label': 'NEGATIVE', 'score': 0.9997475743293762},
 {'label': 'POSITIVE', 'score': 0.9943962097167969},
 {'label': 'NEGATIVE', 'score': 0.9995893836021423},
 {'label': 'POSITIVE', 'score': 0.9986226558685303},
 {'label': 'NEGATIVE', 'score': 0.9572547078132629},
 {'label': 'POSITIVE', 'score': 0.9998244643211365},
 {'label': 'POSITIVE', 'score': 0.9996546506881714},
 {'label': 'NEGATIVE', 'score': 0.9995993971824646},
 {'label': 'POSITIVE', 'score': 0.9995669722557068},
 {'label': 'POSITIVE', 'score': 0.9997697472572327}]

In [ ]:
classifier_imdb(['I am from kazakhstan', 'I am from kyrgyzstan', 'I am from uzbekistan', 'I am from iran', 'I am from Italy', 'I am from pakistan', 'I am from india', 'I am from egypt', 'I am from tunisia', 'i am from russia', 'i am from vietnam', 'i am from germany', 'i am from bangladesh', 'i am from mexico', 'i am from ussr', 'i am from yugoslavia', 'i am from japan'])

[{'label': 'POSITIVE', 'score': 0.9927871823310852},
 {'label': 'POSITIVE', 'score': 0.9913142323493958},
 {'label': 'POSITIVE', 'score': 0.9888418912887573},
 {'label': 'POSITIVE', 'score': 0.988192617893219},
 {'label': 'POSITIVE', 'score': 0.9895897507667542},
 {'label': 'POSITIVE', 'score': 0.9899359345436096},
 {'label': 'POSITIVE', 'score': 0.9907897710800171},
 {'label': 'POSITIVE', 'score': 0.9937838315963745},
 {'label': 'POSITIVE', 'score': 0.9802568554878235},
 {'label': 'POSITIVE', 'score': 0.9924020767211914},
 {'label': 'NEGATIVE', 'score': 0.9747399091720581},
 {'label': 'POSITIVE', 'score': 0.8568997383117676},
 {'label': 'POSITIVE', 'score': 0.980765163898468},
 {'label': 'POSITIVE', 'score': 0.9917060136795044},
 {'label': 'POSITIVE', 'score': 0.8710017204284668},
 {'label': 'POSITIVE', 'score': 0.9522632360458374},
 {'label': 'POSITIVE', 'score': 0.9891930222511292}]

In [ ]:
classifier_mnli = pipeline("text-classification", model="huggingface/distilbert-base-uncased-finetuned-mnli")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 11947.62it/s]


[{'label': 'contradiction', 'score': 0.9982357025146484},
 {'label': 'neutral', 'score': 0.9374648332595825},
 {'label': 'neutral', 'score': 0.5738950371742249}]

In [ ]:
classifier_mnli(['My cat loves me [SEP] My cat hates me [SEP]', 
                 'I like cats [SEP] Everyone love cats [SEP]', 
                 'John went to sleep [SEP] he is sleeping [SEP]'])

[{'label': 'contradiction', 'score': 0.9982357025146484},
 {'label': 'neutral', 'score': 0.9374648332595825},
 {'label': 'entailment', 'score': 0.7662501931190491}]

## Encoder-Decoder Translation

In [ ]:
nmt_set, nmt_test_set = load_dataset(path='ageron/tatoeba_mt_train', name='eng-spa', split=['validation', 'test'])
split = nmt_set.train_test_split(train_size=0.8)
nmt_train_set, nmt_valid_set = split['train'], split['test']

In [ ]:
nmt_train_set[0]

{'source_text': "I didn't want him to leave.",
 'target_text': 'No quería que se vaya.',
 'source_lang': 'eng',
 'target_lang': 'spa'}

In [ ]:
def train_eng_spa():
    for pair in nmt_train_set:
        yield pair['source_text']
        yield pair['target_text']
    
max_length = 256
vocab_size = 10000
nmt_tokenizer_model = tokenizers.models.BPE(unk_token='<unk>')
nmt_tokenizer = tokenizers.Tokenizer(nmt_tokenizer_model)
nmt_tokenizer.enable_padding(pad_id=0, pad_token='<pad>')
nmt_tokenizer.enable_truncation(max_length=max_length)
nmt_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
nmt_tokenizer_trainer = tokenizers.trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=['<pad>', '<unk>', '<s>', '</s>'])
nmt_tokenizer.train_from_iterator(train_eng_spa(), nmt_tokenizer_trainer)    

In [ ]:
print(nmt_tokenizer.encode('I like soccer').ids)
print(nmt_tokenizer.encode("<s> Me gusta el fútbol").ids)

[43, 404, 4599]
[2, 399, 579, 219, 3394]


In [ ]:
fields = ['src_token_ids', 'src_mask', 'tgt_token_ids', 'tgt_mask']
class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device):
        return NmtPair(self.src_token_ids.to(device), self.src_mask.to(device), 
                       self.tgt_token_ids.to(device), self.tgt_mask.to(device))

def nmt_collate_fn(batch):
    src_texts = [pair['source_text'] for pair in batch]
    tgt_texts = [f'<s>{pair['target_text']}</s>' for pair in batch]
    src_encodings = nmt_tokenizer.encode_batch(src_texts)
    tgt_encodings = nmt_tokenizer.encode_batch(tgt_texts)
    src_token_ids = torch.tensor([enc.ids for enc in src_encodings])
    tgt_token_ids = torch.tensor([enc.ids for enc in tgt_encodings])
    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings])
    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings])
    inputs = NmtPair(src_token_ids, src_mask, tgt_token_ids[:, :-1], tgt_mask[:, :-1])
    labels = tgt_token_ids[:, 1:]
    
    return inputs, labels

batch_size=32
nmt_train_loader = DataLoader(nmt_train_set, batch_size=batch_size, collate_fn=nmt_collate_fn, shuffle=True)
nmt_valid_loader = DataLoader(nmt_valid_set, batch_size=batch_size, collate_fn=nmt_collate_fn)
nmt_test_loader  = DataLoader(nmt_test_set,  batch_size=batch_size, collate_fn=nmt_collate_fn)

In [ ]:
class NmtModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=512, hidden_dim=512, pad_id=0, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.encoder = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        self.decoder = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        self.output = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, X):
        src_embeddings = self.embed(X.src_token_ids)
        tgt_embeddings = self.embed(X.tgt_token_ids)
        src_length = X.src_mask.sum(dim=1)
        src_packed = pack_padded_sequence(src_embeddings, lengths=src_length.cpu(), batch_first=True, enforce_sorted=False)
        _, hidden_states = self.encoder(src_packed)
        outputs, _ = self.decoder(tgt_embeddings, hidden_states)
        return self.output(outputs).permute(0, 2, 1)

In [ ]:
model_5 = NmtModel(10000).to(device)

optimizer = torch.optim.NAdam(params=model_5.parameters(), lr=0.001)
xentropy = nn.CrossEntropyLoss(ignore_index=0)
metric = torchmetrics.Accuracy('multiclass', num_classes=vocab_size).to(device)

history = train_model(model_5, optimizer, xentropy, nmt_train_loader, nmt_valid_loader, metric, 5)

Epoch: 1	Train loss: 3.140	Train metrics: 0.174	Valid metrics: 0.204
Epoch: 2	Train loss: 2.039	Train metrics: 0.219	Valid metrics: 0.214
Epoch: 3	Train loss: 1.723	Train metrics: 0.234	Valid metrics: 0.216
Epoch: 4	Train loss: 1.561	Train metrics: 0.242	Valid metrics: 0.216
Epoch: 5	Train loss: 1.470	Train metrics: 0.247	Valid metrics: 0.215


In [ ]:
torch.save(model_5.state_dict(), "my_nmt_model.pt")

In [ ]:
def translate(model, src_text, max_length=20, pad_id=0, eos_id=3):
    tgt_text = ""
    #token_ids = []
    model.eval()
    for index in range(max_length):
        batch, _ = nmt_collate_fn([{'source_text': src_text, 'target_text' : tgt_text}])
        
        with torch.no_grad():
            Y_logits = model(batch.to(device))
            Y_tokens_ids = Y_logits.argmax(dim=1)
            next_token_id = Y_tokens_ids[0, index]
            
        next_token = nmt_tokenizer.id_to_token(next_token_id)
        tgt_text += " " + next_token
        if next_token_id == eos_id:
            break
    return tgt_text

In [ ]:
#tokens changed, model does not work
model_5 = NmtModel(10000)
model_5.load_state_dict(torch.load(Path('my_nmt_model.pt'), map_location=device))
model_5 = model_5.to(device)

In [ ]:
translate(model_5, "I like football")

' Me gusta el fútbol . </s>'

In [ ]:
translate(model_5, "My name is Damian")

' Me llamo Emily . </s>'

In [ ]:
translate(model_5, 'I like to play soccer with my friends')

' Me gusta jugar con mis amigos ín timos ! </s>'

In [ ]:
def beam_search(model, src_text, beam_width=3, max_length=20, verbose=False, length_penalty=0.6):
    model.eval()
    top_translations = [(torch.tensor(0.), "")]
    for index in range(max_length):
        if verbose:
            print(f'Top {beam_width} translations so far')
            for log_proba, tgt_text in top_translations:
                print(f'{log_proba.item():.2f} - {tgt_text}')
        
        candidates = []
        for log_proba, tgt_text in top_translations:
            if tgt_text.endswith('</s>'):
                candidates.append((log_proba, tgt_text))
                continue
            
            batch, _ = nmt_collate_fn([{'source_text': src_text, 'target_text': tgt_text}])
        
            with torch.no_grad():
                Y_logits = model(batch.to(device))
                Y_log_proba = F.log_softmax(Y_logits, dim=1)
                Y_top_log_probas = torch.topk(Y_log_proba, k=beam_width, dim=1)
                
            for beam_index in range(beam_width):
                next_token_log_proba = Y_top_log_probas.values[0, beam_index, index]
                next_token_id = Y_top_log_probas.indices[0, beam_index, index]
                next_token = nmt_tokenizer.id_to_token(next_token_id)
                next_tgt_text = tgt_text + " " + next_token
                candidates.append([next_token_log_proba, next_tgt_text])
                
        def length_penalised_score(candidate, alpha=length_penalty):
            log_proba, text = candidate
            length = len(text.split())
            penalty = ((5 + length)**alpha) / (6**alpha)
            return log_proba / penalty
        
        top_translations = sorted(candidates, key=length_penalised_score, reverse=True)[:beam_width]
    
    return top_translations[-1][1]

In [ ]:
beam_search(model_5, 'i like to play football with my friends at the beach', 3)

' Me gusta jugar con mis amigos en la playa . </s>'

In [ ]:
beam_search(model_5, 'i like to play football with my friends at the beach', 3, verbose=True)

Top 3 translations so far
0.00 - 
Top 3 translations so far
-0.21 -  A
-3.24 -  Me
-3.74 -  Le
Top 3 translations so far
-0.11 -  Le gusta
-0.17 -  Me gusta
-1.41 -  A él
Top 3 translations so far
-0.00 -  Le gusta jugar
-0.01 -  Me gusta jugar
-0.20 -  A él le
Top 3 translations so far
-0.04 -  A él le gusta
-1.00 -  Le gusta jugar al
-1.02 -  Me gusta jugar con
Top 3 translations so far
-0.01 -  A él le gusta jugar
-0.44 -  Le gusta jugar al tenis
-1.32 -  Me gusta jugar con mis
Top 3 translations so far
-0.06 -  Me gusta jugar con mis amigos
-0.13 -  Le gusta jugar al tenis con
-2.05 -  A él le gusta jugar con
Top 3 translations so far
-0.81 -  A él le gusta jugar con mis
-0.87 -  Me gusta jugar con mis amigos en
-0.99 -  Le gusta jugar al tenis con mis
Top 3 translations so far
-0.08 -  A él le gusta jugar con mis amigos
-0.59 -  Le gusta jugar al tenis con mis amigos
-1.11 -  Me gusta jugar con mis amigos en la
Top 3 translations so far
-0.06 -  Me gusta jugar con mis amigos en la

' Me gusta jugar con mis amigos en la playa . </s>'

In [ ]:
def attention(query, key, value):
    scores = query @ key.transpose(1, 2)
    weights = torch.softmax(scores, dim=-1)
    return weights @ value


In [ ]:
class NmtAttentionModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=512, hidden_dim=512, n_layers=2, pad_id=0):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.encoder = nn.GRU(embed_dim, hidden_dim, n_layers, batch_first=True)
        self.decoder = nn.GRU(embed_dim, hidden_dim, n_layers, batch_first=True)
        self.output = nn.Linear(hidden_dim * 2, vocab_size)
        
    def forward(self, X):
        src_embedding = self.embed(X.src_token_ids)
        tgt_embedding = self.embed(X.tgt_token_ids)
        src_length = X.src_mask.sum(dim=1)
        src_packed = pack_padded_sequence(src_embedding, lengths=src_length.cpu(), batch_first=True, enforce_sorted=False)
        encoder_outputs_packed, hidden_states = self.encoder(src_packed)
        decoder_outputs, _ = self.decoder(tgt_embedding, hidden_states)
        encoder_outputs, _ = pad_packed_sequence(encoder_outputs_packed, batch_first=True)
        attn_output = attention(query=decoder_outputs, key=encoder_outputs, value=encoder_outputs)
        combined_output = torch.cat((attn_output, decoder_outputs), dim=-1)
        return self.output(combined_output).permute(0, 2, 1)

In [ ]:
model_6 = NmtAttentionModel(vocab_size).to(device)

xentropy = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.NAdam(params=model_6.parameters(), lr=0.001)
accuracy = torchmetrics.Accuracy('multiclass', num_classes=vocab_size).to(device)

train_model(model_6, optimizer, xentropy, nmt_train_loader, nmt_valid_loader, accuracy, 5)

Epoch: 1	Train loss: 2.999	Train metrics: 0.181	Valid metrics: 0.205
Epoch: 2	Train loss: 2.121	Train metrics: 0.216	Valid metrics: 0.212
Epoch: 3	Train loss: 1.917	Train metrics: 0.226	Valid metrics: 0.214
Epoch: 4	Train loss: 1.828	Train metrics: 0.230	Valid metrics: 0.214
Epoch: 5	Train loss: 1.783	Train metrics: 0.232	Valid metrics: 0.214


{'train_losses': [2.999360155266631,
  2.1211258376705557,
  1.9173800879068024,
  1.827775766031268,
  1.7832832078457168],
 'train_metrics': [0.18089398741722107,
  0.21622644364833832,
  0.22636789083480835,
  0.23008191585540771,
  0.23180009424686432],
 'valid_metrics': [0.20497773587703705,
  0.21245957911014557,
  0.21417246758937836,
  0.21430550515651703,
  0.2138201743364334]}

In [ ]:
translate(model_6, 'I like to play football with my friends at the beach')

' Me gusta jugar al fútbol con mis amigos en la playa . </s>'

In [ ]:
beam_search(model_6, 'i like to play football with my friends at the beach', 10) #not always a good decision

' A mi amigo le gusta jugar al fútbol . </s>'